In [ ]:
import os
import yaml
import polars as pl
import numpy as np
from tqdm import tqdm

from scripts import get_correlations

In [ ]:
config_path = "/home/dnanexus/ukbgym/config_wgs.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)


all_annotation_list = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
if rare_variant_annotations_dict:
    for category in rare_variant_annotations_dict.values():
        all_annotation_list.extend(category)

all_annotation_list = list(set(all_annotation_list))
len(all_annotation_list)

## Merge pos-neg split annotations with other annotations

In [ ]:
config_path = "/home/dnanexus/ukbgym/config_wgs_cadd.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)


pos_neg_annos_list = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
if rare_variant_annotations_dict:
    for category in rare_variant_annotations_dict.values():
        pos_neg_annos_list.extend(category)

pos_neg_annos_list = list(set(pos_neg_annos_list))
len(pos_neg_annos_list)

In [ ]:
del_annos = list(set([an[:-4] for an in pos_neg_annos_list]))
len(del_annos)

In [ ]:
burdens_path = "/home/dnanexus/data_dir/burdens"

all_annos_files = os.listdir(burdens_path + '/55_small_genes_onlySNP_cadd_annotations')

for gene_file in tqdm(all_annos_files):
    print(f"Processing {gene_file}...")

    bdf = pl.read_parquet(
        burdens_path + f'/55_small_genes_onlySNP_cadd_annotations/{gene_file}'
    ).filter(~pl.col('annotation').is_in(del_annos))

    pndf = pl.read_parquet(
        burdens_path + f'/55_small_genes_onlySNP_cadd_annotations_posnegsplit/{gene_file}'
    )

    pl.concat([bdf, pndf]).lazy().sink_parquet(
        burdens_path + f'/55_small_genes_onlySNP_cadd_annotations_posnegsplit_merged/{gene_file}'
    )


## Compute correlations